# 1-Minute Permuted MNIST agent — starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ml-arena/competition-baseline/blob/main/permuted_mnist/agent_baseline.ipynb)

A minimal, runnable baseline for **1-Minute Permuted MNIST**
([#8](https://ml-arena.com/viewcompetition/8)). Each episode you get 60,000 labelled
and 10,000 unlabelled images with pixels **and** labels randomly permuted, and **1
minute** (CPU only) to train and predict. Score = test accuracy.

> This template predicts **random labels**. Replace `train`/`predict` with a fast model
> (a linear/MLP classifier on flattened pixels trains well within the minute).

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk==0.3.0" numpy scikit-learn   # installs the `mlarena` package

import mlarena

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page -> API Keys)
COMPETITION_ID = 8

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Define your agent

The next cell writes **`agent.py`** to disk with `%%writefile`. We upload the whole file
(so its `import`s come along) — that is what the worker runs. Keep it **self-contained**:
it is executed on its own, so it may not import the environment or other notebook cells,
only public packages available in the runtime.

The env calls `train(X_train, y_train)` then `predict(X_test)` each episode. Because
the data crosses a JSON boundary it arrives as plain Python lists — wrap with
`np.asarray(...)` before any array math. `predict` must return a **JSON-serializable
list** of integer labels (one per test image). `__init__` must work with no arguments.

In [ ]:
%%writefile agent.py
import numpy as np


class Agent:
    def __init__(self, output_dim=10, seed=0):
        self.output_dim = output_dim
        self.rng = np.random.RandomState(seed)

    def train(self, X_train, y_train):
        X_train = np.asarray(X_train)
        y_train = np.asarray(y_train)
        # TODO: fit a fast model here (see the last cell). Random agent does not learn.
        pass

    def predict(self, X_test):
        X_test = np.asarray(X_test)
        preds = self.rng.randint(0, self.output_dim, size=X_test.shape[0])
        return preds.tolist()   # NumPy arrays are not JSON-serializable

## 2. (optional) Try it locally

In [ ]:
import numpy as np
from agent import Agent

agent = Agent()
agent.train(np.zeros((5, 784)).tolist(), [0, 1, 2, 3, 4])
print('preds:', agent.predict(np.zeros((3, 784)).tolist()))

## 3. Submit

In [ ]:
# Uploads agent.py (with its imports intact), then creates + deploys the attachment.
result = client.submit(competition_id=COMPETITION_ID, files=["agent.py"])
print(result)

# Stream status until the run reaches a terminal state (deploy -> queue -> run -> scored).
for line in client.tail_logs(COMPETITION_ID, result["attache_agent_id"]):
    print(line)

## 4. Leaderboard

In [ ]:
client.leaderboard(COMPETITION_ID)

## 5. Where to go from here

Pixel permutation destroys spatial structure (so convnets don't help) and label
permutation means you must **learn from the labelled set every episode** — no priors.
A strong-yet-fast baseline that fits in 60s on CPU is a linear classifier on flattened,
normalized pixels — replace `train`/`predict` with, e.g.:

```python
from sklearn.linear_model import LogisticRegression

def train(self, X_train, y_train):
    X = np.asarray(X_train).reshape(len(X_train), -1) / 255.0
    self.clf = LogisticRegression(max_iter=200, n_jobs=-1).fit(X, np.asarray(y_train).ravel())

def predict(self, X_test):
    X = np.asarray(X_test).reshape(len(X_test), -1) / 255.0
    return self.clf.predict(X).tolist()
```

From there: a small MLP (`sklearn.neural_network.MLPClassifier` or PyTorch), PCA to
speed up, or subsampling the training set to stay inside the time budget.